In [1]:
import os, glob, shutil
from PIL import Image

os.makedirs('/kaggle/working/dps', exist_ok=True)
!cp -r /kaggle/input/datasets/aratisutar/dps-code/diffusion-posterior-sampling-main/. /kaggle/working/dps/
os.makedirs('/kaggle/working/dps/models', exist_ok=True)
shutil.copy('/kaggle/input/datasets/aratisutar/dps-weights-imagenet/imagenet256.pt',
            '/kaggle/working/dps/models/imagenet256.pt')

os.makedirs('/kaggle/working/dps/motionblur', exist_ok=True)
open('/kaggle/working/dps/motionblur/__init__.py','w').close()
open('/kaggle/working/dps/motionblur/motionblur.py','w').write(
    'class Kernel:\n    def __init__(self,*a,**k):\n        raise NotImplementedError("stub")\n')

ROOT = '/kaggle/working/dps/data'

def prep_tail(level, files, keep_from=5, n=10):
    d = f'{ROOT}/L{level}'; os.makedirs(d, exist_ok=True)
    chosen, saved = [], 0
    for f in files:
        if saved >= n: break
        try: im = Image.open(f).convert('RGB')
        except Exception: continue
        w, h = im.size
        if min(w, h) < 256: continue
        chosen.append(f); saved += 1
    for j, f in enumerate(chosen[keep_from:]):
        im = Image.open(f).convert('RGB')
        w, h = im.size; s = min(w, h)
        im.crop(((w-s)//2,(h-s)//2,(w+s)//2,(h+s)//2)).resize((256,256), Image.LANCZOS).save(f'{d}/{j:05d}.png')
    return len(chosen), len(os.listdir(d))

# SAME selection as Phase A: first 10 by sorted filename
L0 = sorted(glob.glob('/kaggle/input/datasets/denislukovnikov/ffhq256-images-only/ffhq256/*.png'))[:10]
total, kept = prep_tail(0, L0)
print(f'L0: selected {total}, kept images 5-9 as 5 files -> {kept}')

os.chdir('/kaggle/working/dps')
src = open('configs/super_resolution_config.yaml').read()
open('configs/L0.yaml','w').write(src.replace('root: ./data/samples/', 'root: ./data/L0/'))
print(open('configs/L0.yaml').read())
print('checkpoint:', os.path.getsize('models/imagenet256.pt')//1_000_000, 'MB')
import torch; print('GPU:', torch.cuda.get_device_name(0))

L0: selected 10, kept images 5-9 as 5 files -> 5
conditioning:
  method: ps
  params:
    scale: 0.3

data:
  name: ffhq
  root: ./data/L0/

measurement:
  operator:
    name: super_resolution
    in_shape: !!python/tuple [1, 3, 256, 256]
    scale_factor: 4
  
  noise: 
    name: gaussian
    sigma: 0.05

checkpoint: 2211 MB


GPU: Tesla T4


In [2]:
import subprocess, shutil, os, time

os.chdir('/kaggle/working/dps')
start = time.time()
lvl = 0
src_ids = [f'{i:05d}' for i in range(5)]
dst_ids = [f'{i:05d}' for i in range(5, 10)]
for img in dst_ids:
    os.makedirs(f'/kaggle/working/IN_L{lvl}_{img}', exist_ok=True)

for rep in range(6):
    if all(os.path.exists(f'/kaggle/working/IN_L{lvl}_{d}/{rep:02d}.png') for d in dst_ids):
        continue
    subprocess.run(['python','sample_condition.py',
        '--model_config=configs/imagenet_model_config.yaml',
        '--diffusion_config=configs/diffusion_config.yaml',
        '--task_config=configs/L0.yaml'], check=True, capture_output=True)
    for s, d in zip(src_ids, dst_ids):
        shutil.copy(f'results/super_resolution/recon/{s}.png',
                    f'/kaggle/working/IN_L{lvl}_{d}/{rep:02d}.png')
        shutil.copy(f'results/super_resolution/label/{s}.png',
                    f'/kaggle/working/truth_IN_L{lvl}_{d}.png')
    print(f'L{lvl} rep {rep}   ({(time.time()-start)/60:.1f} min)')
print('RUN DONE')

L0 rep 0   (139.5 min)


L0 rep 1   (278.9 min)


L0 rep 2   (418.3 min)


L0 rep 3   (557.7 min)


L0 rep 4   (697.0 min)
